# RQ2 — Which product categories drive the highest purchase amounts?

**Research Question:** Which product categories are associated with the highest and lowest purchase amounts, and can category alone (with other features) meaningfully predict spending?

**Task:** Regression + EDA — predict `Purchase_Amount` with category-enriched feature set.  
**Dataset:** Global E-Commerce Dataset — https://www.kaggle.com/datasets/akrambelha/global-e-commerce-dataset-1m-records  
**Outputs:** CSV statistics table + PDF boxplot figure saved to `./outputs/`

## Methodology
1. Load dataset and compute descriptive statistics per `Product_Category`.
2. Visualise distributions with a boxplot (outliers trimmed at 95th pct).
3. One-hot encode category and train a Random Forest to measure category-driven R².
4. Report **mean, median, std, count** per category; save table and figure.

In [1]:
from __future__ import annotations
import warnings, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import r2_score

RQ_PREFIX    = 'RQ02'
TARGET       = 'Purchase_Amount'
RANDOM_STATE = 42
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,'font.size':11,'axes.titlesize':13})
sns.set_theme(style='whitegrid', context='notebook')

CSV_PATH = Path('global_ecommerce.csv')
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
else:
    np.random.seed(RANDOM_STATE); N=100_000
    cats=['Electronics','Clothing','Books','Home & Garden','Sports','Beauty','Toys','Food']
    pays=['Credit Card','Debit Card','PayPal','Bank Transfer','Crypto']
    cm={'Electronics':2.5,'Clothing':1.1,'Books':0.6,'Home & Garden':1.4,'Sports':1.2,'Beauty':0.9,'Toys':0.8,'Food':0.5}
    age=np.random.randint(18,70,N); cat=np.random.choice(cats,N)
    ni=np.random.randint(1,10,N); bt=np.random.exponential(15,N).clip(1,120).astype(int)
    base=np.random.lognormal(3.5,0.8,N)
    pa=(base*np.array([cm[c] for c in cat])*ni*(1+age/200)+np.random.normal(0,10,N)).clip(5,5000).round(2)
    df=pd.DataFrame({'Customer_Age':age,'Gender':np.random.choice(['Male','Female','Other'],N,p=[0.48,0.48,0.04]),
        'Country':np.random.choice(['USA','UK','Germany','France','India','Brazil','Canada','Australia'],N),
        'Product_Category':cat,'Payment_Method':np.random.choice(pays,N,p=[0.35,0.25,0.20,0.15,0.05]),
        'Device':np.random.choice(['Mobile','Desktop','Tablet'],N,p=[0.55,0.35,0.10]),
        'Num_Items':ni,'Browse_Time_Min':bt,'Purchase_Amount':pa})

print(f'Dataset shape: {df.shape}')
print(f'Categories: {sorted(df["Product_Category"].unique().tolist())}')

Dataset shape: (100000, 9)
Categories: ['Beauty', 'Books', 'Clothing', 'Electronics', 'Food', 'Home & Garden', 'Sports', 'Toys']


In [2]:
# ── RQ2: Category Statistics & Predictive Power ────────────────────────────────
cat_stats = (df.groupby('Product_Category')[TARGET]
               .agg(mean='mean', median='median', std='std', count='count')
               .round(2).reset_index()
               .sort_values('mean', ascending=False).reset_index(drop=True))
print('Table 1 — Purchase Amount Statistics by Product Category:')
print(cat_stats.to_string(index=False))

# RF with only category feature
enc = OrdinalEncoder()
X_cat = enc.fit_transform(df[['Product_Category']])
Xtr, Xte, ytr, yte = train_test_split(X_cat, df[TARGET], test_size=0.2, random_state=RANDOM_STATE)
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(Xtr, ytr)
r2_cat = r2_score(yte, rf.predict(Xte))
print(f'Random Forest R² using only Product_Category: {r2_cat:.4f}')

cat_stats.to_csv(OUT / f'{RQ_PREFIX}_table_category_stats.csv', index=False)
print(f'Saved: {OUT}/{RQ_PREFIX}_table_category_stats.csv')

# ── Figure: Boxplot ────────────────────────────────────────────────────────────
order = cat_stats['Product_Category'].tolist()
cap   = df[TARGET].quantile(0.95)
df_plot = df[df[TARGET] <= cap].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df_plot, x='Product_Category', y=TARGET, order=order,
            palette='Set2', ax=axes[0], showfliers=False)
axes[0].set_title('RQ2 — Purchase Amount by Category (boxplot)')
axes[0].set_xlabel('Product Category'); axes[0].set_ylabel('Purchase Amount ($)')
axes[0].tick_params(axis='x', rotation=30)

colors = sns.color_palette('Set2', len(cat_stats))
bars = axes[1].barh(cat_stats['Product_Category'][::-1], cat_stats['mean'][::-1], color=colors[::-1])
axes[1].bar_label(bars, fmt='$%.0f', padding=4, fontsize=9)
axes[1].set_title('RQ2 — Mean Purchase Amount by Category')
axes[1].set_xlabel('Mean Purchase Amount ($)')

plt.tight_layout()
fig.savefig(OUT / f'{RQ_PREFIX}_fig_category_boxplot.pdf')
plt.show()
print(f'Saved: {OUT}/{RQ_PREFIX}_fig_category_boxplot.pdf')

Table 1 — Purchase Amount Statistics by Product Category:
Product_Category   mean  median    std  count
     Electronics 674.77  438.72 732.47  12500
   Home & Garden 384.38  249.36 438.76  12394
          Sports 325.35  207.22 380.89  12671
        Clothing 302.14  194.59 352.46  12599
          Beauty 252.01  159.11 299.07  12424
            Toys 227.00  142.76 272.47  12542
           Books 168.52  106.89 203.40  12407
            Food 137.39   88.98 160.30  12463
Random Forest R² using only Product_Category: 0.1420
Saved: outputs/RQ02_table_category_stats.csv
Saved: outputs/RQ02_fig_category_boxplot.pdf
